# MVP v0.2.2: Diffusion Score Guidance (No Proxy)

**Date:** 2026-03-11  
**Builds on:** MVP v0.2 (BC_Gaussian proxy guidance)

**Key change:** Use the actual target policy's score function for guidance instead of a BC_Gaussian proxy.
This matches what SOPE does in its reference implementation — the diffusion noise prediction network
at t=1 gives ∇_a log π(a|s) directly via `-noise_pred / sigma[1]`.

**Other changes from v0.2:**
- Full 19-dim states (keep quaternions) + 7-dim actions (keep orientation) → transition_dim=26
- Retrain chunk diffusion model on full-dim data (can't reuse v0.1/v0.2 checkpoint)
- `RobomimicDiffusionScorer` wraps the target policy to provide `grad_log_prob()`

**Guidance fix (v2):** The scorer now uses chunk-level UNet evaluation — places actual
chunk actions at correct positions in the 16-step prediction horizon, with proper
frame stacking from two different chunk states. Previously tiled a single action 16x
(OOD input for the temporal UNet) and duplicated states for frame stacking.

**Pipeline:**
1. Load 200 demos in full 26-dim space (no quaternion/orientation dropping)
2. Load oracle values (pre-collected, 50 rollouts)
3. Train chunk diffusion on full-dim demo chunks (10 epochs × 1000 steps)
4. Load target policy + create scorer (diffusion score at t=1)
5. Guided stitching using real score function
6. Evaluate OPE vs oracle

In [ ]:
import sys, os
import importlib
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm
import time

# Project root
PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

# SOPE imports
from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

# Robomimic imports
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

# Our imports — force reload to pick up guidance.py fixes
import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
CKPT_DIR = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-11"
DIFFUSION_SAVE_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v022_fulldim"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DIFFUSION_SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

Full 19-dim states + 7-dim actions = 26-dim transitions. No dimension reduction.

In [2]:
# ── Obs keys (sorted, matching robomimic convention & LowDimConcatEncoder layout) ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

# Full dims — NO reduction
STATE_DIM = 19   # object(10) + eef_pos(3) + eef_quat(4) + gripper_qpos(2)
ACTION_DIM = 7   # full robomimic action space
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 26

# ── Diffusion config ──
CHUNK_SIZE = 4
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)   # T=4 supports 2 downsample levels
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False  # x0-prediction (much better per D2.1)

# ── Training config ──
TRAIN_EPOCHS = 10
TRAIN_STEPS_PER_EPOCH = 1000
BATCH_SIZE = 128
LR = 3e-4

# ── Oracle config ──
ORACLE_JSON = CKPT_DIR / "oracle_50.json"

# ── OPE config ──
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60
GAMMA = 1.0

# ── Reward ──
CUBE_Z_INDEX = 2  # index 2 in the 19-dim latent (object key, cube_pos z)
LIFT_THRESHOLD = 0.84

# ── Guidance config ──
GUIDANCE_SCALES = [0.01, 0.05, 0.1, 0.5, 1.0]  # finer sweep than v0.2
NORMALIZE_GRAD = True  # SOPE default

print(f"state_dim={STATE_DIM}, action_dim={ACTION_DIM}, transition_dim={TRANSITION_DIM}")
print(f"Guidance scales to sweep: {GUIDANCE_SCALES}")

state_dim=19, action_dim=7, transition_dim=26
Guidance scales to sweep: [0.01, 0.05, 0.1, 0.5, 1.0]


## Step 0: Load Demo Data (Full Dims)

Load 200 human demos. Keep all 19 state dims and all 7 action dims.

In [3]:
def load_robomimic_demos_full(hdf5_path, obs_keys):
    """Load robomimic demos — full dims, no reduction."""
    data = []
    all_states_list = []
    all_actions_list = []
    
    with h5py.File(hdf5_path, "r") as f:
        demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
        print(f"Found {len(demo_keys)} demos")
        
        for dk in tqdm(demo_keys, desc="Loading demos"):
            demo = f["data"][dk]
            obs_arrays = [demo["obs"][k][:] for k in obs_keys]
            state = np.concatenate(obs_arrays, axis=-1).astype(np.float32)  # (T, 19)
            actions = demo["actions"][:].astype(np.float32)                 # (T, 7)
            rewards = demo["rewards"][:].astype(np.float32)
            
            episode = {
                "states": state[:-1],
                "actions": actions[:-1],
                "rewards": rewards[:-1],
                "next-states": state[1:],
            }
            data.append(episode)
            all_states_list.append(state)
            all_actions_list.append(actions)
    
    all_states = np.concatenate(all_states_list, axis=0)
    all_actions = np.concatenate(all_actions_list, axis=0)
    total_transitions = sum(len(ep["states"]) for ep in data)
    print(f"Loaded {len(data)} episodes, {total_transitions} transitions")
    print(f"State shape: {data[0]['states'].shape}, Action shape: {data[0]['actions'].shape}")
    return data, all_states, all_actions

offline_data, all_states, all_actions = load_robomimic_demos_full(DEMO_HDF5, OBS_KEYS)

Found 200 demos


Loading demos:   0%|          | 0/200 [00:00<?, ?it/s]

Loading demos:  48%|████▊     | 95/200 [00:00<00:00, 944.07it/s]

Loading demos:  96%|█████████▌| 191/200 [00:00<00:00, 949.74it/s]

Loading demos: 100%|██████████| 200/200 [00:00<00:00, 946.20it/s]

Loaded 200 episodes, 9466 transitions
State shape: (58, 19), Action shape: (58, 7)


In [4]:
# ── Normalization stats ──
norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)

norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)

normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

print(f"Norm mean shape: {norm_mean.shape}")
print(f"State mean: {norm_mean[:5]}...")
print(f"Action mean: {norm_mean[STATE_DIM:]}") 

Norm mean shape: (26,)
State mean: [ 0.00185354 -0.00087743  0.8247457  -0.00146249 -0.0012618 ]...
Action mean: [ 0.1724759   0.00581449 -0.16882926  0.00308029  0.00513104  0.0114907
 -0.40761432]


## Step 1: Load Oracle Values

In [5]:
with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)

oracle_returns = np.array(oracle_data["returns"])
oracle_value = float(oracle_data["mean_return"])
oracle_success_rate = float(np.mean(oracle_returns > 0.5))

print(f"Oracle V^π = {oracle_value:.4f} ± {np.std(oracle_returns):.4f}")
print(f"Oracle success rate: {oracle_success_rate*100:.1f}%")

Oracle V^π = 0.5400 ± 0.4984
Oracle success rate: 54.0%


## Step 2: Train Chunk Diffusion (Full 26-dim)

Must retrain since v0.1/v0.2 used 15-dim (reduced). Now using full 26-dim transitions.

In [6]:
# ── Build chunks from demo data ──
def make_chunks(offline_data, chunk_size, state_dim, action_dim):
    """Convert episodes to SOPE-style [state, action] chunks."""
    chunks = []
    for ep in offline_data:
        states = ep["states"]   # (T, state_dim)
        actions = ep["actions"] # (T, action_dim)
        T = len(states)
        transitions = np.concatenate([states, actions], axis=-1)  # (T, 26)
        for start in range(0, T - chunk_size + 1):
            chunk = transitions[start:start + chunk_size]  # (chunk_size, 26)
            chunks.append(chunk)
    chunks = np.array(chunks, dtype=np.float32)
    print(f"Created {len(chunks)} chunks of size {chunk_size} from {len(offline_data)} episodes")
    return chunks

chunks = make_chunks(offline_data, CHUNK_SIZE, STATE_DIM, ACTION_DIM)
print(f"Chunks shape: {chunks.shape}")  # (N, 4, 26)

# Normalize chunks
chunks_norm = (chunks - norm_mean) / norm_std
chunks_tensor = torch.tensor(chunks_norm, dtype=torch.float32, device=device)

Created 8866 chunks of size 4 from 200 episodes
Chunks shape: (8866, 4, 26)


In [ ]:
# ── Load pre-trained diffusion model (skip retraining — only guidance.py changed) ──
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE,
    transition_dim=TRANSITION_DIM,
    dim=BASE_DIM,
    dim_mults=DIM_MULTS,
    attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model,
    horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn,
    unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON,
    loss_type="l2",
    clip_denoised=False,
    action_weight=ACTION_WEIGHT,
    loss_weights=None,
    loss_discount=1.0,
).to(device)

ema = EMA(diffusion_model)

# Load saved checkpoint
ema_ckpt = torch.load(DIFFUSION_SAVE_DIR / "diffusion_model_ema.pt", map_location=device)
ema.ema_model.load_state_dict(ema_ckpt)
model_ckpt = torch.load(DIFFUSION_SAVE_DIR / "diffusion_model.pt", map_location=device)
diffusion_model.load_state_dict(model_ckpt)

n_params = sum(p.numel() for p in diffusion_model.parameters())
train_losses = [0.3175, 0.1367, 0.0905, 0.0655, 0.0503, 0.0427, 0.0359, 0.0331, 0.0291, 0.0271]  # from previous run

print(f"Loaded checkpoint from {DIFFUSION_SAVE_DIR}")
print(f"Diffusion model: {n_params:,} parameters")
print(f"Final training loss (previous run): {train_losses[-1]:.6f}")

## Step 3: Load Target Policy + Create Diffusion Scorer

Use `RobomimicDiffusionScorer` to get `grad_log_prob()` from the actual target policy's score function.
No proxy — this is what SOPE does.

In [8]:
# ── Load target policy and create scorer ──
ckpt = load_checkpoint(CKPT_DIR, ckpt_path=Path("last.pth"))
target_algo = build_algo_from_checkpoint(ckpt, device=str(device))

scorer = RobomimicDiffusionScorer(target_algo, device=str(device), obs_keys=OBS_KEYS)
print(f"Scorer created: state_dim={scorer.state_dim}, action_dim={scorer.action_dim}")
print(f"Sigma at t=1: {scorer.sigma:.6f}")
print(f"Prediction horizon: {scorer.prediction_horizon}")

# Quick sanity check
test_states = torch.randn(4, STATE_DIM, device=device)
test_actions = torch.randn(4, ACTION_DIM, device=device)
test_grad = scorer.grad_log_prob(test_states, test_actions)
print(f"Test grad_log_prob shape: {test_grad.shape}, norm: {test_grad.norm(dim=-1).cpu().numpy()}")


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['object', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []


number of parameters: 6.576359e+07


[11:35:41] INFO     build_algo_from_checkpoint took 0.28 seconds to execute                           ]8;id=143741;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=137246;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\

Scorer created: state_dim=19, action_dim=7
Sigma at t=1: 0.041803
Prediction horizon: 16
Test grad_log_prob shape: torch.Size([4, 7]), norm: [67.80037  75.77043  58.508854 55.885685]


## Step 4: Guided Stitching + OPE

Generate trajectories with diffusion score guidance. The scorer's `grad_log_prob()` 
replaces the BC_Gaussian's `log_prob_extended()` + autograd from v0.2.

In [9]:
def get_initial_states_from_data(offline_data, num_samples, device):
    """Sample initial states from the demo dataset."""
    all_initial = np.array([ep["states"][0] for ep in offline_data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def generate_trajectories_score(diffusion_model, initial_states,
                                normalize_fn, unnormalize_fn,
                                state_dim, action_dim,
                                chunk_size, t_gen, device,
                                scorer=None, action_scale=0.0, normalize_grad=True):
    """Generate trajectories with diffusion score guidance.
    
    Unlike v0.2 which used autograd on BC_Gaussian.log_prob_extended(),
    this uses scorer.grad_log_prob() which evaluates the actual target policy's
    noise prediction network at t=1 (score function).
    """
    guided = (scorer is not None and action_scale > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    
    # Normalize initial states for conditioning
    padded = torch.cat([
        initial_states,
        torch.zeros(batch_size, action_dim, device=device)
    ], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]
    
    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0
    n_iterations = 0
    
    while total_generated < t_gen:
        n_iterations += 1
        steps_remaining = t_gen - total_generated
        shape = (batch_size, chunk_size, transition_dim)
        
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)
        
        for t_diff in reversed(range(diffusion_model.n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)
            
            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)
            
            if guided:
                # Unnormalize to real space
                mean_unnorm = unnormalize_fn(model_mean)
                
                # Extract states and actions
                states_chunk = mean_unnorm[:, :, :state_dim]   # (B, T, 19)
                actions_chunk = mean_unnorm[:, :, state_dim:]  # (B, T, 7)
                
                # Get score from actual target policy (no autograd needed!)
                grad_action = scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                # grad_action: (B, T, action_dim)
                
                # Normalize gradient (SOPE default)
                if normalize_grad:
                    grad_action = grad_action / (grad_action.norm(dim=-1, keepdim=True) + 1e-6)
                
                # Build full guide (zeros for states)
                guide = torch.zeros_like(mean_unnorm)
                guide[:, :, state_dim:] = grad_action
                
                # Apply guidance and re-normalize
                mean_unnorm = mean_unnorm + action_scale * guide
                model_mean = normalize_fn(mean_unnorm)
                model_mean = apply_conditioning(model_mean, conditions, state_dim)
            
            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)
        
        chunk_unnorm = unnormalize_fn(x)
        
        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store
        
        if total_generated >= t_gen:
            break
        
        last_states_norm = x[:, -1, :state_dim]
        conditions = {0: last_states_norm}
    
    label = f"guided(scale={action_scale})" if guided else "unguided"
    print(f"  [{label}] Generated {total_generated} steps in {n_iterations} iterations")
    return all_trajectories.detach().cpu().numpy()


def score_trajectories_gt(trajectories, cube_z_index, threshold, gamma=1.0):
    """Score trajectories using ground-truth Lift reward."""
    B, T, D = trajectories.shape
    returns = np.zeros(B)
    successes = np.zeros(B, dtype=bool)
    for i in range(B):
        gamma_t = 1.0
        for t in range(T):
            reward = 1.0 if trajectories[i, t, cube_z_index] > threshold else 0.0
            returns[i] += reward * gamma_t
            gamma_t *= gamma
            if reward > 0:
                successes[i] = True
    return returns, successes


print("Stitching functions defined.")

Stitching functions defined.


In [10]:
# ── Run stitching: unguided + guided at each scale ──
np.random.seed(42)
torch.manual_seed(42)

initial_states = get_initial_states_from_data(offline_data, NUM_SYNTHETIC_TRAJS, device)
results_by_scale = {}

# 1. Unguided baseline
print("=" * 60)
print("Unguided Stitching (baseline)")
print("=" * 60)
with torch.no_grad():
    unguided_trajs = generate_trajectories_score(
        diffusion_model=ema.ema_model,
        initial_states=initial_states,
        normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM, action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
    )
unguided_returns, unguided_successes = score_trajectories_gt(unguided_trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)
results_by_scale["unguided"] = {
    "trajs": unguided_trajs,
    "returns": unguided_returns,
    "successes": unguided_successes,
    "estimate": float(np.mean(unguided_returns)),
    "std": float(np.std(unguided_returns)),
    "success_rate": float(np.mean(unguided_successes)),
}
print(f"  OPE = {results_by_scale['unguided']['estimate']:.4f}, success = {results_by_scale['unguided']['success_rate']*100:.1f}%\n")

# 2. Guided at each scale (using diffusion score, NOT BC_Gaussian)
for scale in GUIDANCE_SCALES:
    print("=" * 60)
    print(f"Guided Stitching — Diffusion Score (action_scale={scale})")
    print("=" * 60)
    t0 = time.time()
    
    guided_trajs = generate_trajectories_score(
        diffusion_model=ema.ema_model,
        initial_states=initial_states,
        normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM, action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
        scorer=scorer, action_scale=scale, normalize_grad=NORMALIZE_GRAD,
    )
    elapsed = time.time() - t0
    guided_returns, guided_successes = score_trajectories_gt(guided_trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)
    results_by_scale[f"scale={scale}"] = {
        "trajs": guided_trajs,
        "returns": guided_returns,
        "successes": guided_successes,
        "estimate": float(np.mean(guided_returns)),
        "std": float(np.std(guided_returns)),
        "success_rate": float(np.mean(guided_successes)),
        "time": elapsed,
    }
    print(f"  OPE = {results_by_scale[f'scale={scale}']['estimate']:.4f}, "
          f"success = {results_by_scale[f'scale={scale}']['success_rate']*100:.1f}%, "
          f"time = {elapsed:.0f}s\n")

# Summary
print("=" * 60)
print("GUIDANCE SCALE SWEEP — DIFFUSION SCORE (v0.2.2)")
print("=" * 60)
print(f"Oracle V^π = {oracle_value:.4f} (success rate = {oracle_success_rate*100:.1f}%)")
print(f"\n{'Scale':<15} {'OPE Estimate':>13} {'Success Rate':>13} {'Rel Error':>12}")
print("-" * 55)
for name, r in results_by_scale.items():
    rel_err = abs(r["estimate"] - oracle_value) / abs(oracle_value) if oracle_value != 0 else float('inf')
    r["rel_error"] = rel_err
    print(f"{name:<15} {r['estimate']:>13.4f} {r['success_rate']*100:>12.1f}% {rel_err:>11.2%}")

Unguided Stitching (baseline)


  [unguided] Generated 60 steps in 20 iterations
  OPE = 15.7200, success = 98.0%

Guided Stitching — Diffusion Score (action_scale=0.01)


  [guided(scale=0.01)] Generated 60 steps in 20 iterations
  OPE = 15.3000, success = 98.0%, time = 58s

Guided Stitching — Diffusion Score (action_scale=0.05)


  [guided(scale=0.05)] Generated 60 steps in 20 iterations
  OPE = 13.8000, success = 96.0%, time = 58s

Guided Stitching — Diffusion Score (action_scale=0.1)


  [guided(scale=0.1)] Generated 60 steps in 20 iterations
  OPE = 12.6200, success = 96.0%, time = 58s

Guided Stitching — Diffusion Score (action_scale=0.5)


  [guided(scale=0.5)] Generated 60 steps in 20 iterations
  OPE = 16.7600, success = 100.0%, time = 58s

Guided Stitching — Diffusion Score (action_scale=1.0)


  [guided(scale=1.0)] Generated 60 steps in 20 iterations
  OPE = 36.5400, success = 100.0%, time = 58s

GUIDANCE SCALE SWEEP — DIFFUSION SCORE (v0.2.2)
Oracle V^π = 0.5400 (success rate = 54.0%)

Scale            OPE Estimate  Success Rate    Rel Error
-------------------------------------------------------
unguided              15.7200         98.0%    2811.11%
scale=0.01            15.3000         98.0%    2733.33%
scale=0.05            13.8000         96.0%    2455.56%
scale=0.1             12.6200         96.0%    2237.04%
scale=0.5             16.7600        100.0%    3003.70%
scale=1.0             36.5400        100.0%    6666.67%


## Step 5: Visualization + Save Results

In [11]:
# ── Find best scale ──
best_name = None
best_rel_error = float('inf')
for name, r in results_by_scale.items():
    if r["rel_error"] < best_rel_error:
        best_rel_error = r["rel_error"]
        best_name = name

print(f"Best guidance scale: {best_name} (rel error = {best_rel_error:.2%})")

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: OPE by scale
names = list(results_by_scale.keys())
estimates = [results_by_scale[n]["estimate"] for n in names]
stds = [results_by_scale[n]["std"] for n in names]
colors = ["steelblue"] + ["coral"] * len(GUIDANCE_SCALES)

axes[0].bar(range(len(names)), estimates, color=colors, edgecolor="black")
axes[0].errorbar(range(len(names)), estimates, yerr=stds, fmt="none", color="black", capsize=5)
axes[0].axhline(y=oracle_value, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_value:.2f}")
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("OPE Estimate")
axes[0].set_title("OPE vs Guidance Scale (Diffusion Score)")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

# Panel 2: Success rate
success_rates = [results_by_scale[n]["success_rate"] * 100 for n in names]
axes[1].bar(range(len(names)), success_rates, color=colors, edgecolor="black")
axes[1].axhline(y=oracle_success_rate * 100, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_success_rate*100:.0f}%")
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Success Rate (%)")
axes[1].set_title("Synthetic Success Rate")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

# Panel 3: Cube z trajectories
best_trajs = results_by_scale[best_name]["trajs"]
for j in range(min(10, NUM_SYNTHETIC_TRAJS)):
    axes[2].plot(unguided_trajs[j, :, CUBE_Z_INDEX], alpha=0.2, color="steelblue",
                 label="Unguided" if j == 0 else "")
    axes[2].plot(best_trajs[j, :, CUBE_Z_INDEX], alpha=0.2, color="coral",
                 label=f"Best guided ({best_name})" if j == 0 else "")
axes[2].axhline(y=LIFT_THRESHOLD, color="red", linestyle="--", alpha=0.5, label="Lift threshold")
axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Cube z")
axes[2].set_title("Trajectory Quality")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle("MVP v0.2.2: Diffusion Score Guidance (No Proxy)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ope_summary_mvp_v022.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved: {RESULTS_DIR / 'ope_summary_mvp_v022.png'}")

Best guidance scale: scale=0.1 (rel error = 2237.04%)


Figure saved: /home1/reishuen/latent_sope/results/2026-03-11/ope_summary_mvp_v022.png


## Trajectory Visualization

Detailed per-dimension trajectory plots comparing real demos, unguided, and best guided trajectories.

In [ ]:
# ── Dimension labels for the 19-dim state + 7-dim action ──
STATE_LABELS = [
    "cube_x", "cube_y", "cube_z",                           # object pos (0-2)
    "cube_qw", "cube_qx", "cube_qy", "cube_qz",           # object quat (3-6)
    "grip2cube_x", "grip2cube_y", "grip2cube_z",           # gripper-to-cube (7-9)
    "eef_x", "eef_y", "eef_z",                             # eef pos (10-12)
    "eef_qw", "eef_qx", "eef_qy", "eef_qz",              # eef quat (13-16)
    "grip_q0", "grip_q1",                                   # gripper joints (17-18)
]
ACTION_LABELS = [
    "act_dx", "act_dy", "act_dz",                          # position deltas
    "act_drx", "act_dry", "act_drz",                       # orientation deltas
    "act_grip",                                              # gripper
]
ALL_LABELS = STATE_LABELS + ACTION_LABELS

# Key dimensions to visualize (most interpretable)
KEY_STATE_DIMS = {
    "cube_z (lift indicator)": 2,
    "cube_x": 0,
    "eef_z": 12,
    "eef_x": 10,
    "grip2cube_z": 9,
    "gripper_q0": 17,
}
KEY_ACTION_DIMS = {
    "act_dz (vertical)": STATE_DIM + 2,
    "act_dx (lateral)": STATE_DIM + 0,
    "act_grip": STATE_DIM + 6,
}

# Collect some real demo trajectories for comparison
demo_trajs = []
for ep in offline_data[:20]:
    traj = np.concatenate([ep["states"], ep["actions"]], axis=-1)  # (T, 26)
    # Pad/truncate to T_GEN
    if len(traj) >= T_GEN:
        demo_trajs.append(traj[:T_GEN])
    else:
        padded = np.zeros((T_GEN, TRANSITION_DIM), dtype=np.float32)
        padded[:len(traj)] = traj
        padded[len(traj):] = traj[-1]  # repeat last
        demo_trajs.append(padded)
demo_trajs = np.array(demo_trajs)
print(f"Demo trajectories for comparison: {demo_trajs.shape}")

best_trajs = results_by_scale[best_name]["trajs"]
print(f"Best guided: {best_name}")
print(f"Unguided trajs: {unguided_trajs.shape}, Best guided trajs: {best_trajs.shape}")

In [ ]:
# ── Figure 1: Key state dimensions — demos vs unguided vs best guided ──
n_key_states = len(KEY_STATE_DIMS)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

N_SHOW = 10  # trajectories to overlay

for idx, (label, dim) in enumerate(KEY_STATE_DIMS.items()):
    ax = axes[idx]
    # Demos (green)
    for j in range(min(N_SHOW, len(demo_trajs))):
        ax.plot(demo_trajs[j, :, dim], color="green", alpha=0.15,
                label="Demo" if j == 0 else "")
    # Unguided (blue)
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(unguided_trajs[j, :, dim], color="steelblue", alpha=0.25,
                label="Unguided" if j == 0 else "")
    # Best guided (red)
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(best_trajs[j, :, dim], color="coral", alpha=0.25,
                label=f"Guided ({best_name})" if j == 0 else "")
    # Means
    ax.plot(demo_trajs[:N_SHOW, :, dim].mean(axis=0), color="green", linewidth=2, linestyle="--")
    ax.plot(unguided_trajs[:N_SHOW, :, dim].mean(axis=0), color="steelblue", linewidth=2, linestyle="--")
    ax.plot(best_trajs[:N_SHOW, :, dim].mean(axis=0), color="coral", linewidth=2, linestyle="--")
    
    if dim == CUBE_Z_INDEX:
        ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5, label="Lift threshold")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Timestep")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8, loc="upper left")

plt.suptitle("State Trajectories: Demo (green) vs Unguided (blue) vs Best Guided (red)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "traj_states_v022.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'traj_states_v022.png'}")

In [ ]:
# ── Figure 2: Key action dimensions ──
n_key_actions = len(KEY_ACTION_DIMS)
fig, axes = plt.subplots(1, n_key_actions, figsize=(6 * n_key_actions, 4))
if n_key_actions == 1:
    axes = [axes]

for idx, (label, dim) in enumerate(KEY_ACTION_DIMS.items()):
    ax = axes[idx]
    for j in range(min(N_SHOW, len(demo_trajs))):
        ax.plot(demo_trajs[j, :, dim], color="green", alpha=0.15,
                label="Demo" if j == 0 else "")
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(unguided_trajs[j, :, dim], color="steelblue", alpha=0.25,
                label="Unguided" if j == 0 else "")
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(best_trajs[j, :, dim], color="coral", alpha=0.25,
                label=f"Guided ({best_name})" if j == 0 else "")
    # Means
    ax.plot(demo_trajs[:N_SHOW, :, dim].mean(axis=0), color="green", linewidth=2, linestyle="--")
    ax.plot(unguided_trajs[:N_SHOW, :, dim].mean(axis=0), color="steelblue", linewidth=2, linestyle="--")
    ax.plot(best_trajs[:N_SHOW, :, dim].mean(axis=0), color="coral", linewidth=2, linestyle="--")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Timestep")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle("Action Trajectories: Demo (green) vs Unguided (blue) vs Best Guided (red)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "traj_actions_v022.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'traj_actions_v022.png'}")

In [ ]:
# ── Figure 3: Guidance effect across ALL scales — cube_z only ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

all_scale_names = list(results_by_scale.keys())
for idx, name in enumerate(all_scale_names):
    if idx >= len(axes):
        break
    ax = axes[idx]
    trajs = results_by_scale[name]["trajs"]
    est = results_by_scale[name]["estimate"]
    sr = results_by_scale[name]["success_rate"]
    
    # Demo reference (faint)
    for j in range(min(5, len(demo_trajs))):
        ax.plot(demo_trajs[j, :, CUBE_Z_INDEX], color="green", alpha=0.1)
    # Synthetic
    for j in range(min(15, NUM_SYNTHETIC_TRAJS)):
        ax.plot(trajs[j, :, CUBE_Z_INDEX], color="coral", alpha=0.2)
    # Mean
    ax.plot(trajs[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkred", linewidth=2, label="Mean")
    ax.plot(demo_trajs[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkgreen", linewidth=2,
            linestyle="--", label="Demo mean")
    
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(f"{name}\nOPE={est:.2f}, SR={sr*100:.0f}%", fontsize=10)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("cube_z")
    ax.set_ylim([0.78, 0.95])
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8)

# Fill remaining axes
for idx in range(len(all_scale_names), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Cube Z Across All Guidance Scales (red=synthetic, green=demo)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "traj_cubez_all_scales_v022.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'traj_cubez_all_scales_v022.png'}")

In [ ]:
# ── Figure 4: Per-dimension marginal statistics ──
def compute_marginal_stats(trajs, labels):
    """Compute mean, std, min, max per dimension across all trajectories."""
    # trajs: (B, T, D) -> flatten to (B*T, D)
    flat = trajs.reshape(-1, trajs.shape[-1])
    stats = {}
    for i, label in enumerate(labels):
        stats[label] = {
            "mean": flat[:, i].mean(),
            "std": flat[:, i].std(),
            "min": flat[:, i].min(),
            "max": flat[:, i].max(),
        }
    return stats

demo_stats = compute_marginal_stats(demo_trajs, ALL_LABELS)
unguided_stats = compute_marginal_stats(unguided_trajs, ALL_LABELS)
best_guided_stats = compute_marginal_stats(best_trajs, ALL_LABELS)

# Bar chart comparing means
fig, axes = plt.subplots(2, 1, figsize=(18, 8))

# States
state_labels_short = STATE_LABELS
x_pos = np.arange(len(state_labels_short))
width = 0.25
axes[0].bar(x_pos - width, [demo_stats[l]["mean"] for l in state_labels_short],
            width, label="Demo", color="green", alpha=0.7)
axes[0].bar(x_pos,         [unguided_stats[l]["mean"] for l in state_labels_short],
            width, label="Unguided", color="steelblue", alpha=0.7)
axes[0].bar(x_pos + width, [best_guided_stats[l]["mean"] for l in state_labels_short],
            width, label=f"Guided ({best_name})", color="coral", alpha=0.7)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(state_labels_short, rotation=45, ha="right", fontsize=8)
axes[0].set_title("State Dimensions — Marginal Means")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# Actions
action_labels_short = ACTION_LABELS
x_pos_a = np.arange(len(action_labels_short))
axes[1].bar(x_pos_a - width, [demo_stats[l]["mean"] for l in action_labels_short],
            width, label="Demo", color="green", alpha=0.7)
axes[1].bar(x_pos_a,         [unguided_stats[l]["mean"] for l in action_labels_short],
            width, label="Unguided", color="steelblue", alpha=0.7)
axes[1].bar(x_pos_a + width, [best_guided_stats[l]["mean"] for l in action_labels_short],
            width, label=f"Guided ({best_name})", color="coral", alpha=0.7)
axes[1].set_xticks(x_pos_a)
axes[1].set_xticklabels(action_labels_short, rotation=45, ha="right", fontsize=8)
axes[1].set_title("Action Dimensions — Marginal Means")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Marginal Statistics Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "marginal_stats_v022.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'marginal_stats_v022.png'}")

In [ ]:
# ── Figure 5: Return distribution histograms by scale ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, name in enumerate(all_scale_names):
    if idx >= len(axes):
        break
    ax = axes[idx]
    returns = results_by_scale[name]["returns"]
    sr = results_by_scale[name]["success_rate"]
    est = results_by_scale[name]["estimate"]
    
    ax.hist(returns, bins=20, color="coral", edgecolor="black", alpha=0.7)
    ax.axvline(x=oracle_value, color="green", linewidth=2, linestyle="--",
               label=f"Oracle={oracle_value:.2f}")
    ax.axvline(x=est, color="darkred", linewidth=2, linestyle="-",
               label=f"OPE={est:.2f}")
    ax.set_title(f"{name} (SR={sr*100:.0f}%)", fontsize=10)
    ax.set_xlabel("Return")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(len(all_scale_names), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Return Distributions by Guidance Scale", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "return_histograms_v022.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'return_histograms_v022.png'}")

In [12]:
# ── Training loss plot ──
plt.figure(figsize=(8, 3))
plt.plot(train_losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Chunk Diffusion Training (26-dim, {TRAIN_EPOCHS} epochs)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_loss_v022.png", dpi=150)
plt.show()

# ── Save results JSON ──
results_json = {
    "experiment": "MVP_v0.2.2_diffusion_score_guidance",
    "date": "2026-03-11",
    "config": {
        "state_dim": STATE_DIM,
        "action_dim": ACTION_DIM,
        "transition_dim": TRANSITION_DIM,
        "chunk_size": CHUNK_SIZE,
        "n_diffusion_steps": N_DIFFUSION_STEPS,
        "dim_mults": list(DIM_MULTS),
        "predict_epsilon": PREDICT_EPSILON,
        "train_epochs": TRAIN_EPOCHS,
        "train_steps_per_epoch": TRAIN_STEPS_PER_EPOCH,
        "guidance_scales": GUIDANCE_SCALES,
        "normalize_grad": NORMALIZE_GRAD,
        "num_synthetic_trajs": NUM_SYNTHETIC_TRAJS,
        "t_gen": T_GEN,
        "guidance_type": "diffusion_score (RobomimicDiffusionScorer)",
        "score_timestep": scorer.score_timestep,
        "sigma_at_score_t": scorer.sigma,
    },
    "oracle": {
        "value": oracle_value,
        "success_rate": oracle_success_rate,
        "std": float(np.std(oracle_returns)),
    },
    "training": {
        "final_loss": train_losses[-1],
        "all_losses": train_losses,
        "n_chunks": len(chunks),
        "n_params": n_params,
    },
    "results_by_scale": {
        name: {
            "estimate": r["estimate"],
            "std": r["std"],
            "success_rate": r["success_rate"],
            "rel_error": r["rel_error"],
            "returns": r["returns"].tolist(),
        }
        for name, r in results_by_scale.items()
    },
    "best_scale": best_name,
    "best_rel_error": best_rel_error,
}

results_path = RESULTS_DIR / "mvp_v022_results.json"
with open(results_path, "w") as f:
    json.dump(results_json, f, indent=2)

print(f"Results saved: {results_path}")
print(f"\n{'='*60}")
print("MVP v0.2.2 COMPLETE")
print(f"{'='*60}")
print(f"Best: {best_name}, OPE={results_by_scale[best_name]['estimate']:.4f}, rel_error={best_rel_error:.2%}")
print(f"Oracle: V^π={oracle_value:.4f}")
print(f"\nv0.2 (BC_Gaussian proxy, 11+4 dims): best rel_error=25.93%")
print(f"v0.2.2 (diffusion score, 19+7 dims): best rel_error={best_rel_error:.2%}")

Results saved: /home1/reishuen/latent_sope/results/2026-03-11/mvp_v022_results.json

MVP v0.2.2 COMPLETE
Best: scale=0.1, OPE=12.6200, rel_error=2237.04%
Oracle: V^π=0.5400

v0.2 (BC_Gaussian proxy, 11+4 dims): best rel_error=25.93%
v0.2.2 (diffusion score, 19+7 dims): best rel_error=2237.04%


In [ ]:
# ── Save generated rollouts for future use ──
ROLLOUT_SAVE_DIR = RESULTS_DIR / "rollouts_mvp_v022"
ROLLOUT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for name, r in results_by_scale.items():
    safe_name = name.replace("=", "_")
    np.savez(
        ROLLOUT_SAVE_DIR / f"trajs_{safe_name}.npz",
        trajectories=r["trajs"],
        returns=r["returns"],
        successes=r["successes"],
    )
    print(f"Saved {safe_name}: {r['trajs'].shape}")

# Also save initial states for reproducibility
np.save(ROLLOUT_SAVE_DIR / "initial_states.npy", initial_states.cpu().numpy())
print(f"\nAll rollouts saved to {ROLLOUT_SAVE_DIR}")